# Modelado — cribado individual de fracciones sociodemográficas

Este notebook llama al módulo reutilizable `src.modeling.ablacion_fracciones`. Añade una de las 15 fracciones a la vez a la misma base de seis predictores. Compara el valor fijo de 2017 y la serie anual reconstruida en h=2 y h=4, con regresión de casos y clasificación de brote. La selección usa solo validación 2021–2023. Las fracciones anuales de 2018–2024 incorporan información del extremo 2025, por lo que sus métricas son retrospectivas.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_fracciones import ejecutar_ablacion_fracciones

reporte = ejecutar_ablacion_fracciones()

h=2: 30160 filas comunes; 31 variantes
  Selección h=2: {'fija': {'regresion': 'fija__fraccion_rural', 'clasificacion': 'fija__fraccion_sin_seguro'}, 'anual': {'regresion': 'anual__fraccion_agua_red', 'clasificacion': 'anual__fraccion_hogares_lena'}}
h=4: 30030 filas comunes; 31 variantes
  Selección h=4: {'fija': {'regresion': 'fija__fraccion_menores_15', 'clasificacion': 'fija__fraccion_sin_seguro'}, 'anual': {'regresion': 'anual__fraccion_menores_15', 'clasificacion': 'anual__fraccion_alumbrado_red'}}
MLflow: run c05b4eda0b764363857e4eabc55350ae en 'denguekay/ablacion_fracciones' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        folds = resultado['folds_validacion']
        filas.append({'h': int(h), 'variante': variante,
                      'F1_regresion': sum(f['regresion']['alerta_derivada']['f1'] for f in folds) / len(folds),
                      'F1_clasificacion': sum(f['clasificacion']['alerta_directa']['f1'] for f in folds) / len(folds)})
ranking = pd.DataFrame(filas).sort_values(['h', 'variante'])
display(ranking)
for h, bloque in reporte['horizontes'].items():
    print('h =', h, 'selección:', bloque['elegidas_solo_validacion'])
    print('Ganadoras por temporada:', bloque['ganadora_por_temporada'])

,h,variante,F1_regresion,F1_clasificacion
24,2,anual__fraccion_agua_cisterna,0.676792,0.601574
23,2,anual__fraccion_agua_red,0.685257,0.597830
27,2,anual__fraccion_alumbrado_red,0.682107,0.615178
20,2,anual__fraccion_analfabeta_15_mas,0.679634,0.596151
25,2,anual__fraccion_desague_red,0.685109,0.593847
...,...,...,...,...
37,4,fija__fraccion_pared_precaria,0.560462,0.546116
38,4,fija__fraccion_piso_tierra,0.563086,0.554223
32,4,fija__fraccion_rural,0.570776,0.533537
42,4,fija__fraccion_sin_saneamiento,0.565017,0.588705


h = 2 selección: {'fija': {'regresion': 'fija__fraccion_rural', 'clasificacion': 'fija__fraccion_sin_seguro'}, 'anual': {'regresion': 'anual__fraccion_agua_red', 'clasificacion': 'anual__fraccion_hogares_lena'}}
Ganadoras por temporada: {'fija': {'regresion': {'temporada_2021': 'fija__fraccion_rural', 'temporada_2022': 'fija__fraccion_sin_saneamiento', 'temporada_2023': 'base_6'}, 'clasificacion': {'temporada_2021': 'fija__fraccion_sin_seguro', 'temporada_2022': 'base_6', 'temporada_2023': 'base_6'}}, 'anual': {'regresion': {'temporada_2021': 'anual__fraccion_rural', 'temporada_2022': 'anual__fraccion_agua_red', 'temporada_2023': 'base_6'}, 'clasificacion': {'temporada_2021': 'anual__fraccion_hogares_lena', 'temporada_2022': 'anual__fraccion_sin_saneamiento', 'temporada_2023': 'base_6'}}}
h = 4 selección: {'fija': {'regresion': 'fija__fraccion_menores_15', 'clasificacion': 'fija__fraccion_sin_seguro'}, 'anual': {'regresion': 'anual__fraccion_menores_15', 'clasificacion': 'anual__fracci

In [3]:
filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        for fold in resultado['folds_prueba']:
            for enfoque, metrica in (('regresion', fold['regresion']['alerta_derivada']),
                                     ('clasificacion', fold['clasificacion']['alerta_directa'])):
                filas.append({'h': int(h), 'bloque': fold['bloque'], 'variante': variante,
                              'enfoque': enfoque, 'positivos': metrica['positivos'],
                              'VP': metrica['vp'], 'FP': metrica['fp'], 'F1': metrica['f1']})
display(pd.DataFrame(filas).sort_values(['h', 'bloque', 'enfoque', 'variante']))

,h,bloque,variante,enfoque,positivos,VP,FP,F1
15,2,calendario_2025,anual__fraccion_agua_red,clasificacion,3,2,326,0.012085
19,2,calendario_2025,anual__fraccion_hogares_lena,clasificacion,3,1,260,0.007576
3,2,calendario_2025,base_6,clasificacion,3,2,320,0.012308
7,2,calendario_2025,fija__fraccion_rural,clasificacion,3,2,260,0.015094
11,2,calendario_2025,fija__fraccion_sin_seguro,clasificacion,3,1,132,0.014706
14,2,calendario_2025,anual__fraccion_agua_red,regresion,3,0,3,0.000000
18,2,calendario_2025,anual__fraccion_hogares_lena,regresion,3,0,3,0.000000
2,2,calendario_2025,base_6,regresion,3,0,3,0.000000
6,2,calendario_2025,fija__fraccion_rural,regresion,3,0,3,0.000000
10,2,calendario_2025,fija__fraccion_sin_seguro,regresion,3,0,3,0.000000


La interpretación y los límites están en `docs/modeling/ablacion_fracciones.md`. Los bloques 2024 y 2025 se consultaron solo después de fijar las variantes. La selección entre 15 fracciones con tres temporadas es exploratoria.